# 《环境规划与管理》第八讲：US DOE TECHTEST 1.0 快速评估教学实践 (Method 2)
**授课教师**：武汉大学 汤思琪
**授课日期**：2026年10月20日

---
## 教学目标与背景
本实验基于美国能源部（US DOE EERE / AMO / IEDO）发布的 **TECHTEST 1.0**（早期技术技术经济、能源与碳排放启发式评估工具），学习低技术成熟度（TRL 1-4）绿色低碳转型技术的**快速数据录入法（Method 2: Rapid Data Entry Approach）**。

学生将掌握：
1. **功能单位（Functional Unit）与基准技术（Commercial Benchmark）**的对标原则；
2. 基于技术卡片（Technology Cards）的摇篮到大门（Cradle-to-Gate）成本、能耗与温室气体核算；
3. 全国市场渗透潜力与简单投资回收期（Payback Period）的宏观环境规划评价。

In [ ]:
# 1. 环境准备与模块导入
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from pytechtest import TechTestRapidEngine, TechTestReporter, TRLLevel
from pytechtest.models.project import ProjectMetadata, MarketDefinition, FunctionalUnitDefinition
from pytechtest.models.rapid_entry_input import RapidEntryInput, RapidTechCard
import matplotlib.pyplot as plt
print("✅ US DOE TECHTEST 1.0 核心引擎已成功加载！")

## 2. 定义工程项目与市场边界
以武汉某生物基可降解包装薄膜替代传统化石PET薄膜为例：

In [ ]:
# 项目元数据
project = ProjectMetadata(
    title="生物基可降解包装材料产业化环境经济综合评价",
    lead_organization="武汉大学环境规划与管理课题组",
    principal_investigator="课程随堂实践组",
    funding_mechanism="绿色低碳技术早期研发计划",
    trl=TRLLevel.TRL_3
)

# 市场定义（以年替代全国50万吨薄膜市场中的15%为例）
market = MarketDefinition(
    generic_end_use_product="高性能可降解食品包装薄膜",
    annual_us_market_volume=500000.0,  # 吨/年
    market_units="metric tons",
    target_penetration_fraction=0.15,
    market_description="替代传统一次性化石PET膜包装"
)

# 功能单位：以生产 1.0 吨包装薄膜为基准
fu = FunctionalUnitDefinition(
    product_name_new_tech="生物酶促合成薄膜 (New Tech)",
    product_name_benchmark="传统化石PET薄膜 (Benchmark)",
    reference_volume=1.0,
    reference_volume_unit="metric ton",
    equivalence_ratio=1.0
)
print(f"评估基准：{fu.reference_volume} {fu.reference_volume_unit}，目标市场规模：{market.annual_us_market_volume:,.0f} 吨/年")

## 3. 构建新技术与基准技术卡片 (Method 2 Cards)
填报年设计产能、初投资（CapEx）、年化运营成本（OpEx）、摇篮到大门能耗与温室气体强度：

In [ ]:
# 新技术卡片：酶法聚合
card_new = RapidTechCard(
    name="生物酶促直接聚合新工艺",
    product_name="生物基薄膜",
    annual_facility_production=10000.0,  # 年产1万吨
    total_initial_capex=12000000.0,      # 初始设备投资1200万美元
    annualized_capex_with_maint=1800000.0,
    annual_opex=14000000.0,             # 年运营成本1400万美元
    manufacturing_energy_mmbtu=15.0,    # 15 MMBtu/吨
    raw_material_embodied_energy_mmbtu=22.0,
    in_use_energy_mmbtu=0.0,
    manufacturing_emissions_lb=1200.0,  # 1200 lb CO2e/吨
    raw_material_embodied_emissions_lb=1800.0,
    in_use_emissions_lb=0.0
)

# 基准技术卡片：传统化石路线
card_bench = RapidTechCard(
    name="传统石油基PET聚合工艺",
    product_name="化石PET薄膜",
    annual_facility_production=10000.0,
    total_initial_capex=15000000.0,
    annualized_capex_with_maint=2250000.0,
    annual_opex=16500000.0,
    manufacturing_energy_mmbtu=28.0,    # 28 MMBtu/吨
    raw_material_embodied_energy_mmbtu=35.0,
    in_use_energy_mmbtu=0.0,
    manufacturing_emissions_lb=2400.0,  # 2400 lb CO2e/吨
    raw_material_embodied_emissions_lb=3100.0,
    in_use_emissions_lb=0.0
)

rapid_input = RapidEntryInput(
    project=project,
    market=market,
    functional_unit=fu,
    new_technology=card_new,
    commercial_benchmark=card_bench
)

## 4. 执行快速评估引擎并展示仪表盘

In [ ]:
engine = TechTestRapidEngine(rapid_input)
results = engine.run()

reporter = TechTestReporter()
reporter.print_terminal_dashboard(results)

## 5. 教学可视化对比分析 (Matplotlib)

In [ ]:
categories = ["制造成本 ($/吨)", "初级能耗 (MMBtu/吨)", "碳排放 (lb CO2e/吨)"]
bench_vals = [
    results.commercial_benchmark.total_cost_per_reference_volume,
    results.commercial_benchmark.total_energy_per_reference_volume_mmbtu,
    results.commercial_benchmark.total_emissions_per_reference_volume_lb
]
new_vals = [
    results.new_technology.total_cost_per_reference_volume,
    results.new_technology.total_energy_per_reference_volume_mmbtu,
    results.new_technology.total_emissions_per_reference_volume_lb
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ["#d9534f", "#5cb85c"]

for i in range(3):
    axes[i].bar(["基准技术", "绿色新技术"], [bench_vals[i], new_vals[i]], color=colors)
    axes[i].set_title(categories[i], fontsize=12, fontweight="bold")
    axes[i].grid(axis="y", linestyle="--", alpha=0.5)
    pct = (bench_vals[i] - new_vals[i]) / bench_vals[i] * 100
    axes[i].text(0.5, max(bench_vals[i], new_vals[i])*0.85, f"节约/减排: {pct:.1f}%", ha="center", color="#2e6da4", fontweight="bold")

plt.tight_layout()
plt.show()